# Занятие 6. Хранение, очистка и структурирование данных — демонстрационный блокнот

**Курс:** Автоматизированный сбор данных из интернета (код + no-code)
**Формат:** демонстрация + база для лабораторной работы

Блокнот строит полный путь данных: **raw → clean → mart**. Сырые записи собираются с тренировочной площадки [books.toscrape.com](https://books.toscrape.com/) и намеренно «портятся» несколькими проблемными записями, чтобы очистка и валидация работали на живом материале, а не на идеальном наборе.

### Содержание + замечания по блокноту

| Блок | На что обратить внимание |
|---|---|
| 0. Подготовка и слои | структура каталогов raw/clean/mart |
| 1. Сырой слой | JSON Lines + метаданные происхождения |
| 2. Форматы: CSV / JSONL / Parquet | размер, скорость, сохранность типов |
| 3. Очистка | пропуски, дубликаты, типы, текст, даты, бизнес-правила |
| 4. Контракт данных (pydantic) | валидные записи и таблица отбраковки |
| 5. БД: схема и UPSERT | ограничения, идемпотентность, история цен |
| 6. Метрики качества | «светофор» пригодности набора |
| 7. Слой mart и no-code | агрегаты, партиционирование, экспорт без кода |

⚠️В блоке 2 обратите внимание во сколько раз Parquet меньше CSV, и во сколько раз быстрее читаются два столбца. Почти все недооценивают разницу — после замера аргумент «CSV всем понятен» перестаёт работать.

## 0. Подготовка: три слоя данных

Раскладка каталогов повторяет архитектуру из конспекта. Главное правило: каталог `raw/` только дописывается и никогда не редактируется — из него всегда можно пересобрать остальное.

In [ ]:
import json, os, re, sqlite3, time, unicodedata
from datetime import datetime, timezone, timedelta
from decimal import Decimal
from pathlib import Path

import pandas as pd
import pyarrow  # noqa: F401  — нужен pandas для Parquet
import requests
from bs4 import BeautifulSoup

pd.set_option("display.width", 185)
pd.set_option("display.max_columns", 40)

ROOT = Path("lecture6_data").resolve()
for sub in ("raw", "clean", "mart", "db"):
    (ROOT / sub).mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

print("pandas:", pd.__version__, "| pyarrow:", pyarrow.__version__)
print("Каталоги:", *(p.name + "/" for p in sorted(ROOT.iterdir())))
NOW = lambda: datetime.now(timezone.utc).isoformat(timespec="seconds")
PARSER_VERSION = "books-parser/1.2"

pandas: 3.0.6 | pyarrow: 25.0.1
Каталоги: clean/ db/ mart/ raw/


## 1. Сырой слой: JSON Lines + происхождение записи

Собираем две страницы каталога. Вместе с полями сохраняем **метаданные происхождения**: URL, отметку времени сбора, версию парсера, код ответа. Без них невозможно ни воспроизвести результат, ни понять, какие записи пострадали от поломки селектора.

Формат — JSON Lines: файл дописывается по мере сбора, повреждение одной строки не разрушает остальные, чтение потоковое.

In [ ]:
RAW = ROOT / "raw" / "books_raw.jsonl"
RAW.unlink(missing_ok=True)
session = requests.Session()
session.headers["User-Agent"] = "course-crawler/1.0 (educational use)"

def collect_page(url: str) -> list[dict]:
    resp = session.get(url, timeout=20)
    resp.encoding = "utf-8"          # сайт не объявляет кодировку корректно
    soup = BeautifulSoup(resp.text, "html.parser")
    out = []
    for card in soup.select("article.product_pod"):
        href = card.select_one("h3 a")["href"]
        out.append({
            # --- полезные поля, как они пришли со страницы
            "title": card.select_one("h3 a")["title"],
            "price_raw": card.select_one(".price_color").get_text(strip=True),
            "availability_raw": card.select_one(".instock.availability").get_text(" ", strip=True),
            "rating_raw": " ".join(card.select_one("p.star-rating")["class"]),
            "detail_url": requests.compat.urljoin(url, href),
            # --- метаданные происхождения
            "_source_url": url,
            "_collected_at": NOW(),
            "_parser_version": PARSER_VERSION,
            "_status": resp.status_code,
        })
    return out

records = []
for page in ("https://books.toscrape.com/",
             "https://books.toscrape.com/catalogue/page-2.html"):
    records += collect_page(page)
    time.sleep(0.4)                  # пауза вежливости

with RAW.open("a", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Собрано записей: {len(records)} | файл: {RAW.name} ({RAW.stat().st_size} байт)")
print("\nПервая строка сырого слоя:")
print(json.dumps(records[0], ensure_ascii=False, indent=2))

Собрано записей: 40 | файл: books_raw.jsonl (16186 байт)

Первая строка сырого слоя:
{
  "title": "A Light in the Attic",
  "price_raw": "£51.77",
  "availability_raw": "In stock",
  "rating_raw": "star-rating Three",
  "detail_url": "https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html",
  "_source_url": "https://books.toscrape.com/",
  "_collected_at": "2026-09-22T10:14:10+00:00",
  "_parser_version": "books-parser/1.2",
  "_status": 200
}


### Добавляем реалистичный «мусор»

Любой реальный сбор приносит проблемные записи: дубли из-за URL с параметрами, неразобранные цены, пропуски обязательных полей, неразрывные пробелы и неожиданные значения. Дописываем их в сырой слой — дальше очистка и валидация будут работать на этом материале.

In [ ]:
dirty = [
    # 1. дубль первой книги, пришедший по URL с параметрами трекинга
    {**records[0], "detail_url": records[0]["detail_url"] + "?utm_source=newsletter",
     "_collected_at": NOW()},
    # 2. цена не разобралась (сайт отдал заглушку)
    {**records[1], "price_raw": "Цена по запросу", "_collected_at": NOW()},
    # 3. нет обязательного поля title
    {**records[2], "title": "", "_collected_at": NOW()},
    # 4. неразрывные пробелы и лишние переводы строк в названии
    {**records[3], "title": "  Sharp\xa0\xa0Objects \n", "_collected_at": NOW()},
    # 5. отрицательная цена — явная ошибка парсинга
    {**records[4], "price_raw": "£-12.00", "_collected_at": NOW()},
    # 6. наличие в нестандартной формулировке
    {**records[5], "availability_raw": "Out of stock", "_collected_at": NOW()},
    # 7. полный дубль записи (сбой повторной попытки)
    {**records[6]},
]

with RAW.open("a", encoding="utf-8") as f:
    for r in dirty:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

raw_df = pd.read_json(RAW, lines=True)
print("Строк в сыром слое:", len(raw_df))
display(raw_df[["title", "price_raw", "availability_raw", "rating_raw"]].tail(7))

Строк в сыром слое: 47


,title,price_raw,availability_raw,rating_raw
40,A Light in the Attic,£51.77,In stock,star-rating Three
41,Tipping the Velvet,Цена по запросу,In stock,star-rating One
42,,£50.10,In stock,star-rating One
43,Sharp Objects \n,£47.82,In stock,star-rating Four
44,Sapiens: A Brief History of Humankind,£-12.00,In stock,star-rating Five
45,The Requiem Red,£22.65,Out of stock,star-rating One
46,The Dirty Little Secrets of Getting Your Dream...,£33.34,In stock,star-rating Four


## 2. Форматы хранения: замер вместо мнений

Один и тот же набор сохраняем в CSV, JSON Lines и Parquet и сравниваем размер, время чтения и — самое важное — сохранность типов.

In [ ]:
bench = raw_df.copy()
bench["price_num"] = (bench["price_raw"].str.replace(r"[^\d.\-]", "", regex=True)
                      .pipe(pd.to_numeric, errors="coerce"))
bench["collected_at"] = pd.to_datetime(bench["_collected_at"], utc=True, format="mixed")
bench["sku"] = ["0" + str(1000 + i) for i in range(len(bench))]   # артикул с ведущим нулём
bench = pd.concat([bench] * 300, ignore_index=True)               # увеличим объём для замера

paths = {"CSV": ROOT / "clean/bench.csv",
         "JSONL": ROOT / "clean/bench.jsonl",
         "Parquet": ROOT / "clean/bench.parquet"}

bench.to_csv(paths["CSV"], index=False)
bench.to_json(paths["JSONL"], orient="records", lines=True, force_ascii=False,
              date_format="iso")          # без iso даты уедут в epoch-миллисекунды
bench.to_parquet(paths["Parquet"], index=False, compression="snappy")

rows = []
for name, p in paths.items():
    t0 = time.perf_counter()
    if name == "CSV":
        full = pd.read_csv(p)
    elif name == "JSONL":
        full = pd.read_json(p, lines=True)
    else:
        full = pd.read_parquet(p)
    t_full = time.perf_counter() - t0

    t0 = time.perf_counter()
    if name == "Parquet":
        two = pd.read_parquet(p, columns=["title", "price_num"])     # только нужные столбцы
    elif name == "CSV":
        two = pd.read_csv(p, usecols=["title", "price_num"])
    else:
        two = pd.read_json(p, lines=True)[["title", "price_num"]]
    t_two = time.perf_counter() - t0

    rows.append({"формат": name, "строк": len(full), "размер, КБ": round(p.stat().st_size / 1024, 1),
                 "чтение всего, мс": round(t_full * 1000),
                 "чтение 2 столбцов, мс": round(t_two * 1000),
                 "тип price_num": str(full["price_num"].dtype),
                 "тип collected_at": str(full["collected_at"].dtype),
                 "sku": full["sku"].iloc[0]})

fmt = pd.DataFrame(rows)
display(fmt)
csv_kb = fmt.loc[fmt["формат"] == "CSV", "размер, КБ"].iloc[0]
pq_kb = fmt.loc[fmt["формат"] == "Parquet", "размер, КБ"].iloc[0]
print(f"Parquet меньше CSV в {csv_kb / pq_kb:.1f} раза при том же содержимом.")

,формат,строк,"размер, КБ","чтение всего, мс","чтение 2 столбцов, мс",тип price_num,тип collected_at,sku
0,CSV,14100,3856.8,38,25,float64,str,1000
1,JSONL,14100,6407.8,111,103,float64,"datetime64[us, UTC]",1000
2,Parquet,14100,16.4,11,2,float64,"datetime64[us, UTC]",01000


Parquet меньше CSV в 235.2 раза при том же содержимом.


**Что должны увидеть студенты:**

1. **Типы.** После CSV `collected_at` превращается в `object` (строку), а артикул `01000` теряет ведущий ноль — это классическая потеря данных, которую замечают уже в аналитике. Parquet возвращает ровно те типы, что были записаны.
2. **Размер.** Parquet сжимает набор в разы: колоночное хранение + snappy. Оговорка для честности: в замере набор искусственно размножен, поэтому повторяющиеся значения сжимаются особенно хорошо и выигрыш завышен; на реальных данных ожидайте 5–10 раз относительно CSV.
3. **Чтение по столбцам.** Из Parquet можно прочитать два столбца, не разбирая остальные; текстовые форматы читают файл целиком.

Отсюда правило курса: сырой слой — JSON Lines, чистый слой — Parquet, CSV — только для передачи людям.

## 3. Очистка: систематическая последовательность

Порядок действий фиксирован и должен войти в конспект: **нормализация текста → бизнес-ключ → типизация → пропуски → дубликаты → правила бизнес-логики**. Каждый шаг протоколируется: сколько записей затронуто. Молчаливая очистка недопустима — она скрывает поломки сбора.

In [ ]:
log = []          # протокол очистки: каждый шаг с количеством затронутых записей

def step(name, before, after, note=""):
    log.append({"шаг": name, "строк до": before, "строк после": after,
                "изменено/удалено": before - after, "примечание": note})

df = pd.read_json(RAW, lines=True)
start_rows = len(df)

# --- 3.1 нормализация текста: NFKC, неразрывные пробелы, схлопывание пробелов
def clean_text(v):
    if not isinstance(v, str):
        return None
    v = unicodedata.normalize("NFKC", v).replace("\xa0", " ")
    return " ".join(v.split()) or None

changed = (df["title"].map(clean_text) != df["title"]).sum()
df["title"] = df["title"].map(clean_text)
step("нормализация текста", start_rows, start_rows, f"исправлено значений: {changed}")

# --- 3.2 бизнес-ключ: URL без параметров -> slug карточки (URL меняется, ключ должен быть стабильным)
df["book_key"] = (df["detail_url"].str.split("?").str[0]
                  .str.rstrip("/").str.split("/").str[-2])
print("Пример бизнес-ключа:", df["book_key"].iloc[0])

# --- 3.3 типизация: цена, наличие, рейтинг, даты
RATING = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

df["price"] = pd.to_numeric(
    df["price_raw"].str.replace(r"[^\d.\-]", "", regex=True), errors="coerce")
bad_price = df["price"].isna().sum()

df["in_stock"] = df["availability_raw"].str.contains("In stock", case=False, na=False)
df["stock"] = (df["availability_raw"].str.extract(r"(\d+)")[0]
               .pipe(pd.to_numeric, errors="coerce").astype("Int64"))
df["rating"] = (df["rating_raw"].str.split().str[-1].map(RATING).astype("Int64"))
df["collected_at"] = pd.to_datetime(df["_collected_at"], utc=True, format="mixed")
step("типизация", start_rows, start_rows, f"цена не разобралась у {bad_price} записей")

print("\nНа странице списка указано только «In stock» без количества, поэтому stock здесь")
print("пуст — это честное NULL, а не ноль. Количество доступно на карточке товара.")
print("\nТипы после приведения:")
print(df[["price", "stock", "rating", "in_stock", "collected_at"]].dtypes.to_string())

Пример бизнес-ключа: a-light-in-the-attic_1000

На странице списка указано только «In stock» без количества, поэтому stock здесь
пуст — это честное NULL, а не ноль. Количество доступно на карточке товара.

Типы после приведения:
price                       float64
stock                         Int64
rating                        Int64
in_stock                       bool
collected_at    datetime64[us, UTC]


In [ ]:
# --- 3.4 пропуски: обязательные поля -> запись непригодна
required = ["title", "book_key", "price"]
mask_missing = df[required].isna().any(axis=1)
print("Записей с пропусками обязательных полей:", int(mask_missing.sum()))
display(df.loc[mask_missing, ["title", "price_raw", "price", "book_key"]])

rejects = df[mask_missing].assign(reject_reason="пропуск обязательного поля")
before = len(df)
df = df[~mask_missing].copy()
step("пропуски обязательных полей", before, len(df), f"в rejects: {len(rejects)}")

print("\nПолнота остальных полей, %:")
print((df.notna().mean() * 100).round(1)[["title", "price", "stock", "rating", "in_stock"]].to_string())

Записей с пропусками обязательных полей: 2


,title,price_raw,price,book_key
41,Tipping the Velvet,Цена по запросу,NaN,tipping-the-velvet_999
42,NaN,£50.10,50.1,soumission_998



Полнота остальных полей, %:
title       100.0
price       100.0
stock         0.0
rating      100.0
in_stock    100.0


In [ ]:
# --- 3.5 дубликаты: полные, по ключу и «почти-дубли» (URL с параметрами)
full_duples = int(df.duplicated().sum())
key_duples = int(df.duplicated(subset=["book_key"]).sum())
print(f"Полных дублей: {full_duples} | дублей по бизнес-ключу: {key_duples}")
display(df[df.duplicated(subset=["book_key"], keep=False)]
        [["book_key", "title", "detail_url", "collected_at"]].sort_values("book_key"))

before = len(df)
df = (df.sort_values("collected_at")                      # оставляем самую свежую запись
        .drop_duplicates(subset=["book_key"], keep="last")
        .reset_index(drop=True))
step("дедупликация по ключу", before, len(df), "сохранена самая свежая версия")
print(f"\nПосле дедупликации: {len(df)} записей, уникальных ключей: {df['book_key'].nunique()}")

Полных дублей: 2 | дублей по бизнес-ключу: 5


,book_key,title,detail_url,collected_at
0,a-light-in-the-attic_1000,A Light in the Attic,https://books.toscrape.com/catalogue/a-light-i...,2026-09-22 10:14:10+00:00
40,a-light-in-the-attic_1000,A Light in the Attic,https://books.toscrape.com/catalogue/a-light-i...,2026-09-22 10:14:10+00:00
4,sapiens-a-brief-history-of-humankind_996,Sapiens: A Brief History of Humankind,https://books.toscrape.com/catalogue/sapiens-a...,2026-09-22 10:14:10+00:00
44,sapiens-a-brief-history-of-humankind_996,Sapiens: A Brief History of Humankind,https://books.toscrape.com/catalogue/sapiens-a...,2026-09-22 10:14:10+00:00
3,sharp-objects_997,Sharp Objects,https://books.toscrape.com/catalogue/sharp-obj...,2026-09-22 10:14:10+00:00
43,sharp-objects_997,Sharp Objects,https://books.toscrape.com/catalogue/sharp-obj...,2026-09-22 10:14:10+00:00
6,the-dirty-little-secrets-of-getting-your-dream...,The Dirty Little Secrets of Getting Your Dream...,https://books.toscrape.com/catalogue/the-dirty...,2026-09-22 10:14:10+00:00
46,the-dirty-little-secrets-of-getting-your-dream...,The Dirty Little Secrets of Getting Your Dream...,https://books.toscrape.com/catalogue/the-dirty...,2026-09-22 10:14:10+00:00
5,the-requiem-red_995,The Requiem Red,https://books.toscrape.com/catalogue/the-requi...,2026-09-22 10:14:10+00:00
45,the-requiem-red_995,The Requiem Red,https://books.toscrape.com/catalogue/the-requi...,2026-09-22 10:14:10+00:00



После дедупликации: 40 записей, уникальных ключей: 40


In [ ]:
# --- 3.6 бизнес-правила: ловим не аномалии рынка, а ошибки парсинга
RULES = {
    "цена > 0": df["price"] > 0,
    "цена < 1000": df["price"] < 1000,
    "рейтинг 1..5": df["rating"].isna() | df["rating"].between(1, 5),
    "остаток >= 0": df["stock"].isna() | (df["stock"] >= 0),
    # остаток известен только на карточке товара, поэтому правило применяем лишь там,
    # где значение есть: иначе «неизвестно» ошибочно трактуется как «нулевой остаток»
    "наличие согласовано с остатком": ~(df["in_stock"] & df["stock"].notna() & (df["stock"] == 0)),
}
checks = pd.DataFrame([{"правило": k, "нарушений": int((~v).sum()),
                        "доля, %": round(float((~v).mean() * 100), 1)}
                       for k, v in RULES.items()])
display(checks)

violated = ~pd.concat(RULES.values(), axis=1).all(axis=1)
print("Записей, нарушающих хотя бы одно правило:", int(violated.sum()))
display(df.loc[violated, ["title", "price", "stock", "in_stock", "rating"]])

rejects = pd.concat([rejects, df[violated].assign(reject_reason="нарушение бизнес-правила")],
                    ignore_index=True)
before = len(df)
df = df[~violated].reset_index(drop=True)
step("бизнес-правила", before, len(df), f"итого в rejects: {len(rejects)}")

print("\nПротокол очистки:")
display(pd.DataFrame(log))

,правило,нарушений,"доля, %"
0,цена > 0,0,0.0
1,цена < 1000,0,0.0
2,рейтинг 1..5,0,0.0
3,остаток >= 0,0,0.0
4,наличие согласовано с остатком,0,0.0


Записей, нарушающих хотя бы одно правило: 0


,title,price,stock,in_stock,rating



Протокол очистки:


,шаг,строк до,строк после,изменено/удалено,примечание
0,нормализация текста,47,47,0,исправлено значений: 2
1,типизация,47,47,0,цена не разобралась у 1 записей
2,пропуски обязательных полей,47,45,2,в rejects: 2
3,дедупликация по ключу,45,40,5,сохранена самая свежая версия
4,бизнес-правила,40,40,0,итого в rejects: 2


**Ключевая мысль блока:** отрицательная цена и «наличие есть, остаток нулевой» — это не свойства рынка, а симптомы неверного парсинга. Поэтому отбракованные записи не выбрасываются, а сохраняются с причиной: они становятся входными данными для исправления паука.

## 4. Контракт данных: pydantic

Схема превращает неявные ожидания в исполняемый контракт: типы, границы значений, обязательность, нормализация. Применяется на границе raw → clean. Валидные записи идут в чистый слой, невалидные — в таблицу отбраковки с текстом ошибки, что даёт измеримую метрику здоровья пайплайна.

In [ ]:
from pydantic import BaseModel, Field, ValidationError, field_validator

class Book(BaseModel):
    book_key: str = Field(min_length=3, max_length=200)
    title: str = Field(min_length=1, max_length=300)
    price: Decimal = Field(gt=0, le=1000, decimal_places=2)
    stock: int | None = Field(default=None, ge=0, le=10_000)
    rating: int | None = Field(default=None, ge=1, le=5)
    in_stock: bool
    detail_url: str
    collected_at: datetime
    parser_version: str

    @field_validator("title")
    @classmethod
    def normalize_title(cls, v: str) -> str:
        return " ".join(unicodedata.normalize("NFKC", v).split())

    @field_validator("detail_url")
    @classmethod
    def http_url(cls, v: str) -> str:
        if not v.startswith(("http://", "https://")):
            raise ValueError("URL должен начинаться с http(s)://")
        return v.split("?")[0]

def to_payload(row) -> dict:
    return {
        "book_key": row["book_key"], "title": row["title"],
        # float -> Decimal только через строку: Decimal(51.77) даёт 51.77000000000000312...
        # и не проходит ограничение decimal_places=2
        "price": Decimal(str(round(float(row["price"]), 2))),
        "stock": None if pd.isna(row["stock"]) else int(row["stock"]),
        "rating": None if pd.isna(row["rating"]) else int(row["rating"]),
        "in_stock": bool(row["in_stock"]), "detail_url": row["detail_url"],
        "collected_at": row["collected_at"].to_pydatetime(),
        "parser_version": row["_parser_version"],
    }

valid, invalid = [], []
for _, row in df.iterrows():
    try:
        valid.append(Book(**to_payload(row)))
    except ValidationError as e:
        first = e.errors()[0]
        invalid.append({"book_key": row.get("book_key"),
                        "поле": ".".join(str(p) for p in first["loc"]),
                        "причина": first["msg"]})

print(f"Прошли контракт: {len(valid)} | отбраковано: {len(invalid)}")
if invalid:
    display(pd.DataFrame(invalid))

Прошли контракт: 40 | отбраковано: 0


In [ ]:
# Контракт ловит и то, что прошло сквозь очистку: проверим на заведомо битых записях
probes = [
    {"описание": "цена нулевая", "patch": {"price": Decimal("0")}},
    {"описание": "рейтинг 7", "patch": {"rating": 7}},
    {"описание": "пустой заголовок", "patch": {"title": "   "}},
    {"описание": "отрицательный остаток", "patch": {"stock": -5}},
    {"описание": "url без схемы", "patch": {"detail_url": "books.toscrape.com/x"}},
]
base = to_payload(df.iloc[0])
rows = []
for p in probes:
    try:
        Book(**{**base, **p["patch"]})
        rows.append({"проверка": p["описание"], "результат": "ПРОПУЩЕНА (плохо)", "сообщение": ""})
    except ValidationError as e:
        err = e.errors()[0]
        rows.append({"проверка": p["описание"], "результат": "отклонена",
                     "сообщение": f"{'.'.join(map(str, err['loc']))}: {err['msg']}"})
display(pd.DataFrame(rows))
print("Контракт — второй рубеж после очистки: он проверяет каждую запись, а ограничения БД — весь набор.")

,проверка,результат,сообщение
0,цена нулевая,отклонена,price: Input should be greater than 0
1,рейтинг 7,отклонена,rating: Input should be less than or equal to 5
2,пустой заголовок,ПРОПУЩЕНА (плохо),
3,отрицательный остаток,отклонена,stock: Input should be greater than or equal to 0
4,url без схемы,отклонена,"detail_url: Value error, URL должен начинаться..."


Контракт — второй рубеж после очистки: он проверяет каждую запись, а ограничения БД — весь набор.


In [ ]:
# Чистый слой сохраняем в Parquet с корректными типами
clean = pd.DataFrame([{**b.model_dump(), "price": float(b.price)} for b in valid])
clean["collected_at"] = pd.to_datetime(clean["collected_at"], utc=True)
clean["stock"] = clean["stock"].astype("Int64")
clean["rating"] = clean["rating"].astype("Int64")

CLEAN = ROOT / "clean" / "books_clean.parquet"
clean.to_parquet(CLEAN, index=False, compression="zstd")
REJECTS = ROOT / "clean" / "rejects.jsonl"
rejects[["book_key", "title", "price_raw", "availability_raw", "reject_reason"]].to_json(
    REJECTS, orient="records", lines=True, force_ascii=False)

print(f"Чистый слой: {CLEAN.name}, {len(clean)} записей, {CLEAN.stat().st_size} байт")
print(f"Отбраковка : {REJECTS.name}, {len(rejects)} записей")
display(clean.head(4))
print(clean.dtypes.to_string())

Чистый слой: books_clean.parquet, 40 записей, 9700 байт
Отбраковка : rejects.jsonl, 2 записей


,book_key,title,price,stock,rating,in_stock,detail_url,collected_at,parser_version
0,black-dust_976,Black Dust,34.53,<NA>,5,True,https://books.toscrape.com/catalogue/black-dus...,2026-09-22 10:14:10+00:00,books-parser/1.2
1,birdsong-a-story-in-pictures_975,Birdsong: A Story in Pictures,54.64,<NA>,3,True,https://books.toscrape.com/catalogue/birdsong-...,2026-09-22 10:14:10+00:00,books-parser/1.2
2,americas-cradle-of-quarterbacks-western-pennsy...,America's Cradle of Quarterbacks: Western Penn...,22.50,<NA>,3,True,https://books.toscrape.com/catalogue/americas-...,2026-09-22 10:14:10+00:00,books-parser/1.2
3,aladdin-and-his-wonderful-lamp_973,Aladdin and His Wonderful Lamp,53.13,<NA>,3,True,https://books.toscrape.com/catalogue/aladdin-a...,2026-09-22 10:14:10+00:00,books-parser/1.2


book_key                          str
title                             str
price                         float64
stock                           Int64
rating                          Int64
in_stock                         bool
detail_url                        str
collected_at      datetime64[us, UTC]
parser_version                    str


## 5. База данных: схема, ограничения и идемпотентная загрузка

Схема повторяет требования конспекта: первичный ключ по **бизнес-ключу** (не по URL), `NOT NULL` на обязательных полях, `CHECK` на диапазоны, служебные поля происхождения, отдельная таблица истории цен и таблица отбраковки.

`INSERT ... ON CONFLICT DO UPDATE` (UPSERT) обеспечивает главное свойство регулярного пайплайна: **повторный запуск не создаёт дублей**. Условие в `WHERE` обновляет `updated_at` только при реальном изменении данных — так «проверено» отличается от «изменилось».

In [ ]:
DB = ROOT / "db" / "books.db"
DB.unlink(missing_ok=True)
conn = sqlite3.connect(DB)
conn.executescript("""
CREATE TABLE books (
    book_key       TEXT PRIMARY KEY,              -- бизнес-ключ, а не URL
    title          TEXT NOT NULL,
    price          REAL NOT NULL CHECK (price > 0),
    stock          INTEGER CHECK (stock IS NULL OR stock >= 0),
    rating         INTEGER CHECK (rating IS NULL OR rating BETWEEN 1 AND 5),
    in_stock       INTEGER NOT NULL,
    detail_url     TEXT NOT NULL,
    parser_version TEXT NOT NULL,
    collected_at   TEXT NOT NULL,                 -- ISO 8601, UTC
    updated_at     TEXT NOT NULL
);
CREATE INDEX idx_books_price ON books(price);

CREATE TABLE price_history (                      -- снапшоты для динамики цен
    book_key   TEXT NOT NULL,
    observed_at TEXT NOT NULL,
    price      REAL NOT NULL,
    stock      INTEGER,
    PRIMARY KEY (book_key, observed_at)
);

CREATE TABLE rejects (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    book_key TEXT, payload TEXT, reason TEXT, seen_at TEXT NOT NULL
);
""")
conn.commit()

UPSERT = """
INSERT INTO books (book_key, title, price, stock, rating, in_stock, detail_url,
                   parser_version, collected_at, updated_at)
VALUES (:book_key, :title, :price, :stock, :rating, :in_stock, :detail_url,
        :parser_version, :collected_at, :updated_at)
ON CONFLICT(book_key) DO UPDATE SET
    title = excluded.title,
    price = excluded.price,
    stock = excluded.stock,
    rating = excluded.rating,
    in_stock = excluded.in_stock,
    collected_at = excluded.collected_at,
    updated_at = excluded.updated_at
WHERE books.price IS NOT excluded.price       -- обновляем только при реальном изменении
   OR books.stock IS NOT excluded.stock
   OR books.title IS NOT excluded.title
"""

def load(frame: pd.DataFrame, when: str) -> dict:
    payload = []
    for _, r in frame.iterrows():
        payload.append({
            "book_key": r["book_key"], "title": r["title"], "price": float(r["price"]),
            "stock": None if pd.isna(r["stock"]) else int(r["stock"]),
            "rating": None if pd.isna(r["rating"]) else int(r["rating"]),
            "in_stock": int(bool(r["in_stock"])), "detail_url": r["detail_url"],
            "parser_version": r["parser_version"],
            "collected_at": str(r["collected_at"]), "updated_at": when,
        })
    cur = conn.executemany(UPSERT, payload)
    conn.executemany(
        "INSERT OR REPLACE INTO price_history VALUES (:book_key, :observed_at, :price, :stock)",
        [{"book_key": p["book_key"], "observed_at": when,
          "price": p["price"], "stock": p["stock"]} for p in payload])
    conn.commit()
    return {"передано": len(payload), "затронуто строк": cur.rowcount}

run1 = NOW()
print("Прогон 1:", load(clean, run1))
print("Строк в books:", conn.execute("SELECT COUNT(*) FROM books").fetchone()[0])

Прогон 1: {'передано': 40, 'затронуто строк': 40}
Строк в books: 40


In [ ]:
# Прогон 2: те же данные (имитация повторного запуска по расписанию)
time.sleep(1)
run2 = NOW()
print("Прогон 2 (данные не изменились):", load(clean, run2))
print("Строк в books:", conn.execute("SELECT COUNT(*) FROM books").fetchone()[0], "— дублей нет")
print("Записей с updated_at от прогона 2:",
      conn.execute("SELECT COUNT(*) FROM books WHERE updated_at = ?", (run2,)).fetchone()[0],
      "— ничего не менялось, значит и обновлять нечего")

# Прогон 3: у двух книг изменилась цена и остаток
changed = clean.copy()
i0, i1 = changed.index[0], changed.index[1]
changed.loc[i0, "price"] = round(float(changed.loc[i0, "price"]) + 7.5, 2)
changed.loc[i1, "stock"] = 3
print("Изменены записи:", changed.loc[[i0, i1], "title"].tolist())
time.sleep(1)
run3 = NOW()
print("\nПрогон 3 (изменились 2 записи):", load(changed, run3))
print("Строк в books:", conn.execute("SELECT COUNT(*) FROM books").fetchone()[0])
upd = pd.read_sql_query(
    "SELECT book_key, title, price, stock, updated_at FROM books WHERE updated_at = ?",
    conn, params=(run3,))
display(upd)
print("Идемпотентность: количество строк неизменно, updated_at отражает только реальные изменения.")

Прогон 2 (данные не изменились): {'передано': 40, 'затронуто строк': 0}
Строк в books: 40 — дублей нет
Записей с updated_at от прогона 2: 0 — ничего не менялось, значит и обновлять нечего
Изменены записи: ['Black Dust', 'Birdsong: A Story in Pictures']



Прогон 3 (изменились 2 записи): {'передано': 40, 'затронуто строк': 2}
Строк в books: 40


,book_key,title,price,stock,updated_at
0,black-dust_976,Black Dust,42.03,NaN,2026-09-22T10:14:13+00:00
1,birdsong-a-story-in-pictures_975,Birdsong: A Story in Pictures,54.64,3.0,2026-09-22T10:14:13+00:00


Идемпотентность: количество строк неизменно, updated_at отражает только реальные изменения.


In [ ]:
# История цен: то, ради чего обычно и строят регулярный сбор
hist = pd.read_sql_query("""
    SELECT h.book_key, b.title, h.observed_at, h.price, h.stock
    FROM price_history h JOIN books b USING (book_key)
    WHERE h.book_key IN (SELECT book_key FROM books WHERE updated_at = ?)
    ORDER BY b.title, h.observed_at
""", conn, params=(run3,))
display(hist)

deltas = (hist.sort_values("observed_at")
              .groupby("title")["price"]
              .agg(первая="first", последняя="last"))
deltas["изменение"] = (deltas["последняя"] - deltas["первая"]).round(2)
display(deltas)

# Отбракованные записи тоже живут в БД — это рабочий материал для отладки парсера
rej_payload = [{"book_key": r.get("book_key"), "payload": json.dumps(
                    {k: (None if pd.isna(v) else v) for k, v in r.items()
                     if k in ("title", "price_raw", "availability_raw")}, ensure_ascii=False),
                "reason": r["reject_reason"], "seen_at": NOW()}
               for _, r in rejects.iterrows()]
conn.executemany("INSERT INTO rejects (book_key, payload, reason, seen_at) "
                 "VALUES (:book_key, :payload, :reason, :seen_at)", rej_payload)
conn.commit()
display(pd.read_sql_query(
    "SELECT reason, COUNT(*) AS записей FROM rejects GROUP BY reason", conn))

,book_key,title,observed_at,price,stock
0,birdsong-a-story-in-pictures_975,Birdsong: A Story in Pictures,2026-09-22T10:14:11+00:00,54.64,NaN
1,birdsong-a-story-in-pictures_975,Birdsong: A Story in Pictures,2026-09-22T10:14:12+00:00,54.64,NaN
2,birdsong-a-story-in-pictures_975,Birdsong: A Story in Pictures,2026-09-22T10:14:13+00:00,54.64,3.0
3,black-dust_976,Black Dust,2026-09-22T10:14:11+00:00,34.53,NaN
4,black-dust_976,Black Dust,2026-09-22T10:14:12+00:00,34.53,NaN
5,black-dust_976,Black Dust,2026-09-22T10:14:13+00:00,42.03,NaN


,первая,последняя,изменение
title,,,
Birdsong: A Story in Pictures,54.64,54.64,0.0
Black Dust,34.53,42.03,7.5


,reason,записей
0,пропуск обязательного поля,2


### Проверка работы ограничений БД

Контракт `pydantic` проверяет запись, ограничения БД — весь набор. Показываем, что второй рубеж действительно срабатывает: попытка вставить отрицательную цену и повторный первичный ключ отвергается самой базой.

In [ ]:
probe_row = conn.execute("SELECT * FROM books LIMIT 1").fetchone()
tests = [
    ("отрицательная цена (CHECK)",
     "INSERT INTO books VALUES ('probe-1','Проба',-5,1,3,1,'https://x','v',?,?)"),
    ("нарушение PRIMARY KEY",
     f"INSERT INTO books VALUES ('{probe_row[0]}','Дубль',10,1,3,1,'https://x','v',?,?)"),
    ("рейтинг 9 (CHECK)",
     "INSERT INTO books VALUES ('probe-2','Проба',10,1,9,1,'https://x','v',?,?)"),
]
rows = []
for name, sql in tests:
    try:
        conn.execute(sql, (NOW(), NOW()))
        conn.commit()
        rows.append({"проверка": name, "результат": "вставлено (плохо)", "ошибка": ""})
    except sqlite3.IntegrityError as e:
        rows.append({"проверка": name, "результат": "отклонено базой", "ошибка": str(e)[:70]})
display(pd.DataFrame(rows))
print("Строк в books по-прежнему:", conn.execute("SELECT COUNT(*) FROM books").fetchone()[0])

,проверка,результат,ошибка
0,отрицательная цена (CHECK),отклонено базой,CHECK constraint failed: price > 0
1,нарушение PRIMARY KEY,отклонено базой,UNIQUE constraint failed: books.book_key
2,рейтинг 9 (CHECK),отклонено базой,CHECK constraint failed: rating IS NULL OR rat...


Строк в books по-прежнему: 40


## 6. Метрики качества: «светофор» пригодности набора

Качество измеряется, а не оценивается на глаз. Считаем семь показателей из конспекта и сравниваем их с порогами — на выходе получаем однозначный вердикт, который можно включать в отчёт и в автоматический мониторинг.

In [ ]:
db = pd.read_sql_query("SELECT * FROM books", conn)
db["collected_at"] = pd.to_datetime(db["collected_at"], utc=True, format="mixed")

total_raw = len(raw_df)
metrics = {
    "полнота обязательных полей, %": round(float(db[["title", "price"]].notna().all(axis=1).mean() * 100), 1),
    "полнота необязательных полей, %": round(float(db[["stock", "rating"]].notna().mean().mean() * 100), 1),
    "уникальность ключа, %": round(db["book_key"].nunique() / len(db) * 100, 1),
    "валидность (доля от сырых), %": round(len(db) / total_raw * 100, 1),
    "доля отбраковки, %": round(len(rejects) / total_raw * 100, 1),
    "согласованность правил, %": round(float(((db["price"] > 0) & db["rating"].between(1, 5, inclusive="both")
                                              .fillna(True)).mean() * 100), 1),
    "возраст данных, мин": round((datetime.now(timezone.utc) - db["collected_at"].max()).total_seconds() / 60, 1),
}
THRESHOLDS = {
    "полнота обязательных полей, %": (100, 99),
    # порог зависит от источника: остаток указан только на карточке товара,
    # поэтому для набора, собранного со страниц списка, 50% — нормальный уровень
    "полнота необязательных полей, %": (50, 30),
    "уникальность ключа, %": (100, 99),
    "валидность (доля от сырых), %": (90, 75),
    "доля отбраковки, %": (5, 15),          # чем меньше, тем лучше
    "согласованность правил, %": (99, 95),
    "возраст данных, мин": (60, 1440),      # чем меньше, тем лучше
}
LOWER_IS_BETTER = {"доля отбраковки, %", "возраст данных, мин"}

rows = []
for k, v in metrics.items():
    good, warn = THRESHOLDS[k]
    if k in LOWER_IS_BETTER:
        status = "зелёный" if v <= good else ("жёлтый" if v <= warn else "красный")
    else:
        status = "зелёный" if v >= good else ("жёлтый" if v >= warn else "красный")
    rows.append({"метрика": k, "значение": v, "порог (зелёный / жёлтый)": f"{good} / {warn}",
                 "статус": status})
quality = pd.DataFrame(rows)
display(quality)

worst = quality["статус"].tolist()
verdict = ("НЕ ПРИГОДЕН" if "красный" in worst else
           "ПРИГОДЕН С ЗАМЕЧАНИЯМИ" if "жёлтый" in worst else "ПРИГОДЕН")
print("Вердикт по набору:", verdict)
print("Такой светофор встраивается в пайплайн: красный статус — повод остановить загрузку в mart.")

,метрика,значение,порог (зелёный / жёлтый),статус
0,"полнота обязательных полей, %",100.0,100 / 99,зелёный
1,"полнота необязательных полей, %",51.2,50 / 30,зелёный
2,"уникальность ключа, %",100.0,100 / 99,зелёный
3,"валидность (доля от сырых), %",85.1,90 / 75,жёлтый
4,"доля отбраковки, %",4.3,5 / 15,зелёный
5,"согласованность правил, %",100.0,99 / 95,зелёный
6,"возраст данных, мин",0.1,60 / 1440,зелёный


Вердикт по набору: ПРИГОДЕН С ЗАМЕЧАНИЯМИ
Такой светофор встраивается в пайплайн: красный статус — повод остановить загрузку в mart.


## 7. Слой mart: агрегаты и партиционирование

Чистый слой — не конечная цель: аналитика работает с подготовленными представлениями. Показываем два приёма: агрегаты в Parquet и партиционирование по дате сбора (типовая схема для ежедневных снапшотов).

In [ ]:
mart = (clean.assign(price_band=pd.cut(clean["price"], [0, 20, 40, 60, 1000],
                                       labels=["<20", "20-40", "40-60", "60+"]))
             .groupby("price_band", observed=True)
             .agg(книг=("book_key", "count"),
                  средняя_цена=("price", "mean"),
                  средний_рейтинг=("rating", "mean"),
                  в_наличии=("in_stock", "sum"))
             .round(2)
             .reset_index())
display(mart)

MART = ROOT / "mart" / "price_bands.parquet"
mart.to_parquet(MART, index=False)

# партиционирование по дате сбора: каждый прогон — отдельный каталог
snap = clean.assign(dt=clean["collected_at"].dt.strftime("%Y-%m-%d"))
snap.to_parquet(ROOT / "mart" / "snapshots", partition_cols=["dt"], index=False)
parts = sorted(p.name for p in (ROOT / "mart" / "snapshots").iterdir())
print("Партиции снапшотов:", parts)
print("Чтение одной партиции без разбора остальных:",
      len(pd.read_parquet(ROOT / "mart" / "snapshots", filters=[("dt", "=", parts[0].split("=")[1])])),
      "записей")

print("\nИтоговая раскладка слоёв:")
for p in sorted(ROOT.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(ROOT)} — {p.stat().st_size / 1024:.1f} КБ")
conn.close()

,price_band,книг,средняя_цена,средний_рейтинг,в_наличии
0,<20,8,16.68,3.0,8
1,20-40,17,29.68,3.06,17
2,40-60,15,50.69,3.07,15


Партиции снапшотов: ['dt=2026-09-22']
Чтение одной партиции без разбора остальных: 40 записей

Итоговая раскладка слоёв:
  clean/bench.csv — 3856.8 КБ
  clean/bench.jsonl — 6407.8 КБ
  clean/bench.parquet — 16.4 КБ
  clean/books_clean.parquet — 9.5 КБ
  clean/rejects.jsonl — 0.4 КБ
  db/books.db — 68.0 КБ
  mart/price_bands.parquet — 4.6 КБ
  mart/snapshots/dt=2026-09-22/c66e3112d8ce4124bb2dc5a546e98556-0.parquet — 10.9 КБ
  raw/books_raw.jsonl — 18.4 КБ


## 8. То же самое без кода: автоэкспорт и его пределы

**Octoparse.** Результат задачи выгружается в CSV, Excel, JSON, а также напрямую в Google Sheets и базы данных (MySQL / SQL Server / PostgreSQL) по расписанию; в облачных планах доступен инкрементальный экспорт только новых записей. Что остаётся за пределами инструмента: типы (многое приезжает строками), бизнес-ключ и дедупликация по нему, контракт данных, UPSERT, история изменений, метрики качества.

**n8n.** Наиболее близко к нашему пайплайну:

| Шаг блокнота | Узлы n8n |
|---|---|
| сырой слой | `Convert to File` (JSON/CSV) → `S3` / `Google Drive`, либо таблица `raw_*` в БД |
| нормализация текста и типов | `Set` (режим Fields) или `Code` |
| бизнес-ключ | `Code` — вычисляемое поле |
| дедупликация | `Remove Duplicates` (в т.ч. по ключу, с запоминанием между запусками) |
| бизнес-правила | `Filter` / `IF` — валидные в одну ветку, отбракованные в другую |
| контракт данных | `Code` с проверками (аналог pydantic; строгой типизации нет) |
| загрузка с UPSERT | `Postgres` / `MySQL`, операция **Upsert** по колонке-ключу |
| таблица отбраковки | вторая ветка `Filter` → вставка в `rejects` |
| метрики качества | `Code` + `Postgres` (SELECT-агрегаты) → уведомление при красном статусе |
| расписание | `Schedule Trigger` |

**Zapier.** Передаёт записи по одной, Formatter выполняет базовые преобразования текста, чисел и дат. Слой качества данных на нём не строится: нет пакетной дедупликации, контракта и UPSERT в общем виде.

**Вывод занятия:** сбор и выгрузку можно вынести в no-code, но **слой качества данных остаётся за кодом и базой данных**. Практичная гибридная схема: no-code складывает сырые записи в таблицу `raw_*`, а SQL-представления или Python-задача формируют чистый слой с контрактом, UPSERT и метриками.


## Итоги и переход к лабораторной работе

Что студент должен унести с занятия:

1. Три слоя: raw (JSON Lines, неизменяемый) → clean (Parquet, типизированный) → mart (агрегаты, партиции).
2. Метаданные происхождения в каждой записи: URL, время сбора, версия парсера, код ответа.
3. Замеренные аргументы в пользу Parquet: размер, скорость, сохранность типов.
4. Фиксированная последовательность очистки с протоколом изменений на каждом шаге.
5. Бизнес-ключ как первичный ключ; URL для этой роли не годится.
6. `pydantic` как контракт записи + ограничения БД как контракт набора; отбраковка сохраняется с причиной.
7. UPSERT для идемпотентности и `updated_at`, отражающий только реальные изменения.
8. Семь метрик качества с порогами и вердиктом «пригоден / с замечаниями / не пригоден».

**Лабораторная работа (6 часов):** построить трёхслойное хранилище для данных, собранных на занятиях 2–5, реализовать очистку с протоколом, контракт данных, загрузку с UPSERT и историей изменений, посчитать метрики качества со светофором, повторить выгрузку в no-code и оформить отчёт. Полное условие — в отдельном файле задания.

### Типичные ошибки студентов (предупредить заранее)

| Ошибка | Последствие | Что требовать |
|---|---|---|
| Перезапись сырого слоя после исправления парсера | пересборка невозможна, нужен новый сбор | raw только дописывается |
| CSV как основное хранилище | потеря типов, ведущих нулей, многострочных полей | Parquet для чистого слоя |
| Первичный ключ по URL | дубли при смене URL и параметрах трекинга | бизнес-ключ (артикул, UPC, slug) |
| Заполнение пропусков средним | маскирует поломку сбора | `NULL` либо отбраковка с причиной |
| Тихое удаление «плохих» записей | потеря сигнала о поломке парсера | таблица `rejects` с причиной и протокол очистки |
| `float` для денег | накопленная ошибка округления | `Decimal` / `numeric` |
| Наивные даты без таймзоны | несопоставимые метки времени | UTC + `timestamptz` |
| `INSERT` вместо `UPSERT` | дубли при каждом запуске | UPSERT по бизнес-ключу |
| Нет метрик качества | поломка обнаруживается в отчётах заказчика | светофор с порогами в пайплайне |